# 03 - Gaussian Naive Bayes: Pelatihan Model

## 5. Metode Komputasi Soft Computing yang Digunakan

**Naive Bayes (Gaussian Naive Bayes).** Karena seluruh delapan atribut
prediktor berupa data numerik kontinu, varian Gaussian dipakai --
setiap atribut diasumsikan berdistribusi normal pada tiap kelas.
Tahapan pengerjaan pada notebook ini meliputi:

1. Penghitungan **probabilitas prior** tiap kelas, `P(diabetes)` dan
   `P(tidak diabetes)`, dari data latih.
2. Estimasi **distribusi normal** (mean dan variansi) tiap atribut,
   untuk masing-masing kelas.
3. Penerapan **Teorema Bayes** untuk menghitung probabilitas posterior
   `P(kelas | atribut)` pada data uji, dengan asumsi independensi
   antar atribut.
4. Pemilihan kelas dengan probabilitas posterior tertinggi sebagai
   hasil prediksi.

Evaluasi kuantitatif (akurasi, precision, recall, F1, confusion
matrix) dilakukan terpisah pada `04_Evaluation.ipynb`. Notebook ini
fokus pada pelatihan model dan pembuktian bahwa keempat tahap di atas
benar-benar terjadi di dalam `GaussianNB`, bukan sekadar memanggil
`.fit()` sebagai kotak hitam.


In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_data
from src.pipeline import train_and_predict, split_features_target, RANDOM_STATE, TEST_SIZE
from src.preprocessing import ZERO_AS_MISSING_COLUMNS

DATA_PATH = PROJECT_ROOT / "data" / "diabetes.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "diabetes_clean.csv"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

df_mentah = load_data(DATA_PATH)
df_bersih = pd.read_csv(CLEAN_PATH)

print(f"Data tanpa cleaning : {df_mentah.shape}")
print(f"Data dengan cleaning: {df_bersih.shape}")
print(f"random_state={RANDOM_STATE}, test_size={TEST_SIZE} (identik untuk kedua skenario)")


Data tanpa cleaning : (768, 9)
Data dengan cleaning: (768, 9)
random_state=42, test_size=0.2 (identik untuk kedua skenario)


## Melatih Dua Skenario

Model dilatih dua kali dengan fungsi `train_and_predict` yang sama
persis dari `src/pipeline.py`: sekali pada data mentah (nilai 0 masih
apa adanya), sekali pada data bersih (hasil `02_Preprocessing`).
Satu-satunya perbedaan antara kedua skenario adalah data masukannya,
sehingga perbandingan performa pada `04_Evaluation` adil.


In [2]:
hasil_tanpa = train_and_predict(df_mentah)
hasil_dengan = train_and_predict(df_bersih)

model_tanpa = hasil_tanpa["model"]
model_dengan = hasil_dengan["model"]

print("Model tanpa cleaning : dilatih pada", hasil_tanpa["X_train"].shape[0], "sampel latih")
print("Model dengan cleaning: dilatih pada", hasil_dengan["X_train"].shape[0], "sampel latih")


Model tanpa cleaning : dilatih pada 614 sampel latih
Model dengan cleaning: dilatih pada 614 sampel latih


## 5.1 Probabilitas Prior Tiap Kelas

`GaussianNB.class_prior_` menyimpan `P(kelas)` yang dihitung langsung
dari proporsi kelas pada data latih. Dicocokkan di bawah dengan
perhitungan manual (`value_counts(normalize=True)`) untuk membuktikan
keduanya identik -- tidak ada "kotak hitam" di sini.


In [3]:
prior_model = pd.Series(model_dengan.class_prior_, index=model_dengan.classes_, name="prior_dari_model")
prior_manual = hasil_dengan["y_train"].value_counts(normalize=True).sort_index()
prior_manual.name = "prior_manual"

tabel_prior = pd.concat([prior_model, prior_manual], axis=1)
tabel_prior.index.name = "Outcome"
tabel_prior["cocok"] = np.isclose(tabel_prior["prior_dari_model"], tabel_prior["prior_manual"])
tabel_prior


,prior_dari_model,prior_manual,cocok
Outcome,,,
0,0.651466,0.651466,True
1,0.348534,0.348534,True


## 5.2 Estimasi Distribusi Normal (Mean dan Variansi) per Atribut per Kelas

Untuk tiap kelas, `GaussianNB` mengestimasi mean (`theta_`) dan
variansi (`var_`) tiap atribut dari data latih -- inilah parameter
distribusi normal yang dipakai sebagai *likelihood* pada Teorema
Bayes.


In [4]:
fitur = list(hasil_dengan["X_train"].columns)
distribusi = pd.DataFrame({
    "mean_kelas_0": model_dengan.theta_[0],
    "var_kelas_0": model_dengan.var_[0],
    "mean_kelas_1": model_dengan.theta_[1],
    "var_kelas_1": model_dengan.var_[1],
}, index=fitur)
distribusi.round(2)


,mean_kelas_0,var_kelas_0,mean_kelas_1,var_kelas_1
Pregnancies,3.32,8.87,4.76,13.53
Glucose,110.39,548.52,142.88,870.22
BloodPressure,70.44,147.30,75.34,141.14
SkinThickness,26.98,70.48,32.83,77.31
Insulin,113.14,4571.39,186.00,7115.23
BMI,30.77,40.04,35.53,44.00
DiabetesPedigreeFunction,0.44,0.09,0.55,0.13
Age,31.28,134.43,37.26,126.60


Perhatikan `Glucose`: mean pada kelas diabetes (`mean_kelas_1`) jauh
lebih tinggi daripada kelas tidak diabetes (`mean_kelas_0`) --
konsisten dengan pemisahan kelas yang terlihat pada histogram di
`01_EDA`, dan menjelaskan mengapa `Glucose` menjadi atribut paling
diskriminatif bagi model.


## 5.3 Penerapan Teorema Bayes: Menghitung Posterior Secara Manual

Bagian ini menghitung probabilitas posterior `P(kelas | atribut)`
untuk satu sampel data uji **tanpa memanggil `predict_proba`**,
langsung dari definisi Teorema Bayes dengan asumsi independensi
(karena itu disebut *naive*):

$$P(y \mid x_1, \dots, x_8) \propto P(y) \prod_{i=1}^{8} P(x_i \mid y)$$

dengan tiap `P(x_i | y)` adalah fungsi kepadatan probabilitas (PDF)
normal memakai `mean_kelas` dan `var_kelas` pada tabel di atas.
Dihitung dalam ruang log agar stabil secara numerik, lalu
dinormalisasi kembali ke probabilitas dengan softmax. Hasilnya
dicocokkan dengan `model.predict_proba()` untuk membuktikan bahwa
inilah persis yang dilakukan `GaussianNB` secara internal.


In [5]:
def log_pdf_normal(x, mean, var):
    return -0.5 * np.log(2 * np.pi * var) - ((x - mean) ** 2) / (2 * var)


def posterior_manual(sample, model):
    log_prior = np.log(model.class_prior_)
    log_likelihood = np.array([
        log_pdf_normal(sample.values, model.theta_[k], model.var_[k]).sum()
        for k in range(len(model.classes_))
    ])
    joint = log_prior + log_likelihood
    # softmax untuk normalisasi ke probabilitas
    joint_stabil = joint - joint.max()
    probs = np.exp(joint_stabil)
    probs = probs / probs.sum()
    return probs


sampel = hasil_dengan["X_test"].iloc[0]
label_sebenarnya = hasil_dengan["y_test"].iloc[0]

posterior_manual_hasil = posterior_manual(sampel, model_dengan)
posterior_sklearn = model_dengan.predict_proba(sampel.to_frame().T)[0]

print("Sampel data uji (indeks 0):")
print(sampel.to_string())
print()
print(f"Label sebenarnya (Outcome): {label_sebenarnya}")
print()
print(f"{'Kelas':<20}{'Posterior manual':<20}{'Posterior sklearn':<20}")
for kelas, p_manual, p_sklearn in zip(model_dengan.classes_, posterior_manual_hasil, posterior_sklearn):
    nama = "Tidak Diabetes" if kelas == 0 else "Diabetes"
    print(f"{nama:<20}{p_manual:<20.6f}{p_sklearn:<20.6f}")

assert np.allclose(posterior_manual_hasil, posterior_sklearn, atol=1e-6), "Posterior manual tidak cocok dengan sklearn!"
print()
print("Posterior manual dan hasil predict_proba() sklearn identik (selisih < 1e-6).")


Sampel data uji (indeks 0):
Pregnancies                   7.000
Glucose                     159.000
BloodPressure                64.000
SkinThickness                27.000
Insulin                     102.500
BMI                          27.400
DiabetesPedigreeFunction      0.294
Age                          40.000

Label sebenarnya (Outcome): 0

Kelas               Posterior manual    Posterior sklearn   
Tidak Diabetes      0.605357            0.605357            
Diabetes            0.394643            0.394643            

Posterior manual dan hasil predict_proba() sklearn identik (selisih < 1e-6).


## 5.4 Keputusan Kelas

Kelas dengan probabilitas posterior tertinggi dipilih sebagai hasil
prediksi (`argmax`). Baris berikut menunjukkan keputusan model untuk
seluruh data uji, dibandingkan dengan `model.predict()`.


In [6]:
prediksi_manual = np.array([
    model_dengan.classes_[np.argmax(posterior_manual(hasil_dengan["X_test"].iloc[i], model_dengan))]
    for i in range(len(hasil_dengan["X_test"]))
])
prediksi_sklearn = hasil_dengan["y_pred"]

cocok = (prediksi_manual == prediksi_sklearn).all()
print(f"Prediksi manual (argmax posterior) identik dengan model.predict() untuk seluruh {len(prediksi_sklearn)} data uji: {cocok}")


Prediksi manual (argmax posterior) identik dengan model.predict() untuk seluruh 154 data uji: True


## Menyimpan Model dan Artefak Deployment

Model yang dilatih pada data bersih (`model_dengan`) dipakai sebagai
model utama aplikasi deteksi langsung (Streamlit), karena performanya
sedikit lebih baik (dibuktikan pada `04_Evaluation`). Model pada data
mentah tetap disimpan untuk keperluan perbandingan pada
`04_Evaluation`. Urutan kolom fitur turut disimpan agar aplikasi
Streamlit selalu menyusun input dengan urutan yang identik dengan saat
pelatihan.


In [7]:
import json

joblib.dump(model_dengan, MODELS_DIR / "gnb_clean.joblib")
joblib.dump(model_tanpa, MODELS_DIR / "gnb_raw.joblib")

with open(MODELS_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(fitur, f, indent=2)

print("Tersimpan:")
print(" ", MODELS_DIR / "gnb_clean.joblib", "(model utama, dipakai aplikasi Streamlit)")
print(" ", MODELS_DIR / "gnb_raw.joblib", "(pembanding, dipakai 04_Evaluation)")
print(" ", MODELS_DIR / "feature_columns.json", "->", fitur)


Tersimpan:
  d:\diabetes-risk-gnb\models\gnb_clean.joblib (model utama, dipakai aplikasi Streamlit)
  d:\diabetes-risk-gnb\models\gnb_raw.joblib (pembanding, dipakai 04_Evaluation)
  d:\diabetes-risk-gnb\models\feature_columns.json -> ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']


**Lanjut ke `04_Evaluation.ipynb`** untuk evaluasi kuantitatif kedua
model (akurasi, precision, recall, F1, confusion matrix) dan diskusi
kritis mengenai dampak pelanggaran asumsi independensi.
